## EAS 4331/6331 PHYSICAL VOLCANOLOGY


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyMelt as pm 

#### <b> (1) BRIEF INTRODUCTION TO pyMelt </b>
 
pyMelt is a Python library developed by Simon Matthews for computing the melting behavior of Earth's mantle for a variety of lithologies. pyMelt uses melting equations developed by Morgan (2001). 

pyMelt can be installed using pip. For more information on how to install and use pyMelt, see the <a href="https://github.com/simonwmatthews/pyMelt"> pyMelt Github </a>, and <a href="https://pymelt.readthedocs.io/en/latest/index.html"> documentation </a>. pyMelt organization as well as the math implemented are described in <a href="https://www.jvolcanica.org/ojs/index.php/volcanica/article/view/147"> this paper</a>.


#### <b> (2) CREATE ONE OR MORE LITHOLOGY OBJECTS </b>
pyMelt has "lithology" objects which allow us to compute the melting behavior of induvidual lithologies (recall the ultramafic ternary diagram). I have created a lherzolite lithology object below, and plotted the melting curve (solidus) and the liquidus. Similarly, you can create any number of lithology objects and inspect their solidus and liquidus. For a complete list of supported lithologies see <a href="https://pymelt.readthedocs.io/en/latest/tutorial1.html#Lithology-objects"> here</a>.


In [ ]:
l = pm.lithologies.ball.mixed_mantle()

P = np.arange(0, 3, 0.1)
sol = np.zeros_like(P)
liq = np.zeros_like(P)

for i in range(len(P)):
    sol[i] = l.TSolidus(P[i])
    liq[i] = l.TLiquidus(P[i])
    
plt.figure()
plt.plot(sol, P, label="solidus (melting curve)")
plt.plot(liq, P, label="liquidus")
plt.xlabel("temperature (C)")
plt.ylabel("pressure (GPa)")
plt.legend()
plt.show()

#### <b> (3) CREATE A MANTLE OBJECT </b>
A mantle object is made of multiple lithologies in pyMelt, just as how Earth's mantle is heterogeneous. I have created a mantle that is 100% lherzolite below and printed out its density at different temperatures.

You can create a mantle with multiple lithologies by specifying the lithology names and proportions as Python lists (similar to the first line of the code below). 

In [ ]:
man = pm.mantle([l],[1])
print("The bulk density of my mantle is ", man. bulkProperties()["rho"], " g/cc")
print("The bulk density of my mantle at 1600 C is ", man.bulkProperties(T=1600.0, P=1)["rho"], " g/cc")

#### <b> (4) COMPUTE THE MELTING TEMPERATURE OF THE MANTLE </b>
We will primarily look at decompression melting in this exercise. Remember this is the type of melting that occurs when a rock parcel is brought up from greater depths by mantle convection. But first, we will inspect isobaric melting, i.e., melting occuring at a constant pressure. We will also inspect the density of the mantle rock at different temperatures.

In [ ]:
print("My mantle starts melting at ", np.nanmin(man.solidusIntersectionIsobaric(2.0)), " C")


T = np.arange(600, 2500, 10)
rho = []
for t in T:
    rho.append(man.bulkProperties(T=t, P=1)["rho"])
plt.figure()
plt.plot(T, rho)
plt.xlabel("Temperature (C)")
plt.ylabel("Density")
plt.show()

### <b><u> TASK 1 - MANTLE DENSITY</u> (3 pts) </b>
Pick a single- or a multi-lithology mantle of your choice. Plot the change in bulk density as a function of temperature by writing your own code in the cell below; modify the code snippet above as needed to accomplish this. 

<ol>
    <li> In 1-2 sentences, describe the trend in density that you observe. (1pt) </li>
    <li> Change the value of pressure alone and keep your mantle the same. Explore how the density versus temperature trend changes for different pressures. Explain in your own words, how and why the density curve changes when you change pressure. (2 pts) </li>
</ol>


In [ ]:
# your code





***

#### <b> (5) ADIABATIC DECOMPRESSION MELTING </b> 
Next, we will use PyMelt to visualize adiabatic decompression melting in the mantle. I will create a new multi-lithology mantle for this, since I have so far only used a single-lithology mantle. 

In [ ]:
l1 = pm.lithologies.katz.lherzolite()
l2 = pm.lithologies.pertermann.g2()

mantle = pm.mantle([l1, l2], [.6,.4], ["lherzolite", "pyroxenite"])
col = mantle.adiabaticMelt(1200, Pstart=4, Pend=0)
f, a = col.plot()

In my example, the pyroxenite lithology starts melting at at much greater depths than the lherzolite lithology. The melting curve for the pyroxenite is lower than the mantle adiabat between 2.5 and 0 GPa, whereas the lherzolite melting curve is depressed compared to the mantle adiabat only close to the surface. 

#### <b> (6) MELTING AT MID-OCEAN RIDGES </b>
We can assign this melting column to a mid-ocean ridge setting to study decompression melting and crust formation.

In [ ]:
morb = pm.geosettings.spreadingCentre(col)
print("Mid-ocean ridge crustal thickness is "+str(round(morb.tc,2))+" km") 
print("The lithology contributions to the new crust is as follows:")
print(morb.lithology_contributions)

### <b><u> TASK 2 - ADIABATIC DECOMPRESSION MELTING</u> (3 pts) </b>
For this mantle meting task, create a mantle with three lithologies; use any combination of lithologies except the "non-melting harzburgite". 
<ol>
    <li> Compute adiabatic decompression melting column for pressures between 4 GPa and 0 GPa for 1000 and 1300 deg C potential temperatures, and plot them separately. (1 pt) </li>
    <li> Compute the relative contributions of your lithologies to the new crust. Compare your results with the melt fractions of the lithologies (you can inspect the values using <b><i> col.lithologies["lherzolite"].F </i></b>. What commonalities do you observe between the two? What does this tell about how the composition of the crust relates to the mantle lithologies? (2 pts)  </li>
</ol>

In [ ]:
### your code






***

#### <b> (7) ADIABATIC DECOMPRESSION MELTING OF HYDROUS MANTLE </b>
We have been looking at anhydrous lithologies so far. Addition of water depresses the melting curve. We can verify this by creating hydrous lithologies and comparing with results for anhydrous lithologies. 

In [ ]:
# anhydrous single-lithology mantle
anh_l = pm.lithologies.matthews.klb1()
anh_man = pm.mantle([anh_l], [1], ["lherzolite"])

# hydrous mantle
hyd_l = pm.hydrousLithology(anh_l, H2O=0.2)
hyd_man = pm.mantle([hyd_l], [1], ["hyd. lherzolite"])

In [ ]:
anh_col = anh_man.adiabaticMelt(1400, Pstart=8)
hyd_col = hyd_man.adiabaticMelt(1400, Pstart=8)

In [ ]:
fig, ax = plt.subplots(1,2, sharey = "row")
fig.set_size_inches(6,3)

ax[0].plot(anh_col.T, anh_col.P, label='Anhydrous', c='0.7')
ax[0].plot(hyd_col.T, hyd_col.P, label='0.2 wt%, batch', c='C1')

ax[1].plot(anh_col.F, anh_col.P, label='Anhydrous', c='0.7')
ax[1].plot(hyd_col.F, hyd_col.P, label='0.2 wt%, batch', c='C1')

ax[0].set_xlabel('Temperature ($^\circ$C)')
ax[1].set_xlabel('F')
ax[0].set_ylabel('Pressure (GPa)')
ax[0].invert_yaxis()
ax[1].legend()
plt.show()

***
### <b><u> TASK 3 - ADIABATIC DECOMPRESSION MELTING OF HYDROUS LITHOLOGIES </u> (4 pts) </b>
<ol>
    <li> Using the same temperature and pressure values as shown, compute and plot the melting behavior of a single lithology anhydrous mantle, and single-lithology hydrous mantles with water wt % of 0.05 and 0.2 (ignore any warnings that you get) (2 pts) </li>
    <li> Assign your three melting columns to mid-ocean ridge settings and compute the crustal thicknesses for the three cases. Explain the trends in crustal thickness that you observe in the context of melt fraction generated for each case (2 pts) </li>
</ol>

In [ ]:
# your code






***